In [6]:
# =============================================================
#  PUNE NO2 — LIGHTWEIGHT INTERACTIVE MAP (Plotly WebGL)
#
#  WHY THIS DOESN'T CRASH:
#  Old approach: one Python object per pixel × 9 layers = 400k objects in RAM
#                HTML file = 500MB+ → browser crashes
#
#  This approach: Plotly uses WebGL — ALL pixels drawn as ONE
#                 canvas element on the GPU. RAM usage ~100MB.
#                 HTML file = ~5MB. Browser handles it instantly.
#
#  Install:
#    pip install plotly pandas numpy
#
#  Run:
#    python3 create_map_lightweight.py
#
#  Output:
#    map_outputs/pune_no2_map.html   ← open in any browser
# =============================================================

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import os
import json
import warnings
warnings.filterwarnings("ignore")

os.makedirs("map_outputs", exist_ok=True)

# ── FILE PATHS ─────────────────────────────────────────────────
# Script tries each path in order — uses first one that exists
PRED_PATHS = [
    "data/output/all_predictions_500m.csv",
    "map_outputs/all_predictions.csv",
    "data/output/training_data_500m.csv",
    "data/Preprocessed/ml_ready.csv",
    "ml_ready.csv",
]

OUTPUT_HTML = "map_outputs/pune_no2_map.html"

# ── THRESHOLDS ─────────────────────────────────────────────────
WHO_LIMIT   = 25.0
NAAQS_LIMIT = 80.0

# ── AQI SCALE (CPCB India) ─────────────────────────────────────
AQI_BINS   = [0,  40,   80,  120,  180,  250,  999]
AQI_LABELS = ["Good","Satisfactory","Moderate","Poor","Very Poor","Severe"]
AQI_COLORS = ["#00E400","#FFFF00","#FF7E00","#FF0000","#8F3F97","#7E0023"]

SEASON_NAMES = {0:"Winter",1:"Summer",2:"Monsoon",3:"Post-monsoon"}

STATIONS = {
    "Gavalinagar":   (18.63673, 73.82487),
    "Katraj Dairy":  (18.45445, 73.85416),
    "Park Wakad":    (18.59700, 73.76200),
    "SPPU":          (18.52900, 73.85100),
    "Thergaon":      (18.61100, 73.78400),
    "Bhumkar Nagar": (18.59800, 73.77300),
    "Hadapsar":      (18.50300, 73.93900),
    "Panchawati":    (18.53600, 73.82600),
    "Savta Mali":    (18.62800, 73.80600),
    "Nigdi":         (18.65400, 73.78800),
}


# =============================================================
#  STEP 1 — LOAD PREDICTIONS (memory efficient)
# =============================================================

def load_data():
    for path in PRED_PATHS:
        if not os.path.exists(path):
            continue
        print(f"Loading: {path}")

        # Read only columns we need — saves RAM
        needed = ["latitude","longitude","NO2_pred_ugm3",
                  "NO2_ground_ugm3","date","season","month",
                  "sat_lat","sat_lon"]

        all_cols = pd.read_csv(path, nrows=0).columns.tolist()
        use_cols = [c for c in needed if c in all_cols]
        df       = pd.read_csv(path, usecols=use_cols)

        # Rename columns to standard names
        if "sat_lat"         in df.columns: df = df.rename(columns={"sat_lat":"latitude","sat_lon":"longitude"})
        if "NO2_ground_ugm3" in df.columns and "NO2_pred_ugm3" not in df.columns:
            df["NO2_pred_ugm3"] = df["NO2_ground_ugm3"]
        if "season" not in df.columns and "month" in df.columns:
            df["season"] = df["month"].map(
                {12:0,1:0,2:0,3:1,4:1,5:1,6:2,7:2,8:2,9:2,10:3,11:3})

        df = df.dropna(subset=["latitude","longitude","NO2_pred_ugm3"])
        df = df[df["NO2_pred_ugm3"].between(0, 500)]

        print(f"  Rows: {len(df):,}  Unique pixels: {df[['latitude','longitude']].drop_duplicates().shape[0]:,}")
        return df

    raise FileNotFoundError(
        "No prediction file found. Run step2_train_and_predict.py first.\n"
        f"Expected one of:\n" + "\n".join(f"  {p}" for p in PRED_PATHS)
    )


# =============================================================
#  STEP 2 — AGGREGATE (annual mean per unique pixel)
# =============================================================
# WHY: If you have 2.5M rows (339 dates × 7400 pixels),
# we reduce to ~7400 rows by taking the annual mean per pixel.
# Plotly WebGL handles 44,589 points instantly — no need to plot
# every single daily reading.

def aggregate(df):
    annual = (df.groupby(["latitude","longitude"], as_index=False)
                .agg(NO2_pred_ugm3=("NO2_pred_ugm3","mean"))
                .round({"NO2_pred_ugm3":2}))

    # AQI category for each pixel
    annual["aqi_label"] = pd.cut(
        annual["NO2_pred_ugm3"],
        bins=AQI_BINS,
        labels=AQI_LABELS,
        right=False
    ).astype(str)

    # WHO/NAAQS compliance
    annual["status"] = np.where(
        annual["NO2_pred_ugm3"] <= WHO_LIMIT,   "Within WHO",
        np.where(
        annual["NO2_pred_ugm3"] <= NAAQS_LIMIT, "Exceeds WHO",
                                                "Exceeds NAAQS"
    ))

    # Hover text — built once, used by Plotly directly
    annual["hover"] = (
        "<b>NO₂: " + annual["NO2_pred_ugm3"].astype(str) + " µg/m³</b>" +
        "<br>AQI: "    + annual["aqi_label"] +
        "<br>Status: " + annual["status"] +
        "<br>Lat: "    + annual["latitude"].round(5).astype(str) +
        "<br>Lon: "    + annual["longitude"].round(5).astype(str)
    )

    print(f"Annual mean pixels: {len(annual):,}")
    print(f"NO2 range: {annual['NO2_pred_ugm3'].min():.1f} – "
          f"{annual['NO2_pred_ugm3'].max():.1f} µg/m³")
    return annual


def get_seasonal(df):
    if "season" not in df.columns:
        return {}
    seasons = {}
    for s_code, s_name in SEASON_NAMES.items():
        sub = df[df["season"] == s_code]
        if len(sub) == 0:
            continue
        smean = (sub.groupby(["latitude","longitude"], as_index=False)
                    .agg(NO2_pred_ugm3=("NO2_pred_ugm3","mean"))
                    .round({"NO2_pred_ugm3":2}))
        smean["hover"] = (
            f"<b>{s_name} mean: " +
            smean["NO2_pred_ugm3"].astype(str) + " µg/m³</b>" +
            "<br>Lat: " + smean["latitude"].round(5).astype(str) +
            "<br>Lon: " + smean["longitude"].round(5).astype(str)
        )
        seasons[s_name] = smean
    return seasons


def get_exceedance(df):
    exc = (df.groupby(["latitude","longitude"], as_index=False)
             .apply(lambda g: pd.Series({
                 "pct_naaqs": round((g["NO2_pred_ugm3"]>NAAQS_LIMIT).mean()*100, 1),
                 "pct_who":   round((g["NO2_pred_ugm3"]>WHO_LIMIT).mean()*100,   1),
             }))
             .reset_index(drop=True))
    exc["hover"] = (
        "<b>NAAQS exceeded: " + exc["pct_naaqs"].astype(str) + "% of days</b>" +
        "<br>WHO exceeded: " + exc["pct_who"].astype(str) + "% of days" +
        "<br>Lat: " + exc["latitude"].round(5).astype(str) +
        "<br>Lon: " + exc["longitude"].round(5).astype(str)
    )
    return exc


# =============================================================
#  STEP 3 — BUILD PLOTLY FIGURE
# =============================================================

def build_figure(annual, seasons, exceedance, df):
    print("\nBuilding Plotly WebGL figure...")

    # Continuous colour scale matching CPCB AQI
    colorscale = [
        [0.00, "#00E400"],
        [0.20, "#FFFF00"],
        [0.40, "#FF7E00"],
        [0.60, "#FF0000"],
        [0.80, "#8F3F97"],
        [1.00, "#7E0023"],
    ]
    vmax = min(float(annual["NO2_pred_ugm3"].quantile(0.97)), 200)

    fig = go.Figure()

    # ── TRACE 1: Annual mean (default visible) ──────────────────
    # scattermapbox uses WebGL — handles 100k+ points easily
    fig.add_trace(go.Scattermapbox(
        name="Annual mean NO₂",
        lat=annual["latitude"],
        lon=annual["longitude"],
        mode="markers",
        visible=True,
        marker=dict(
            size=6,
            color=annual["NO2_pred_ugm3"],
            colorscale=colorscale,
            cmin=0, cmax=vmax,
            opacity=0.85,
            colorbar=dict(
                title=dict(text="NO₂ (µg/m³)", side="right"),
                thickness=14,
                len=0.7,
                x=1.01,
                tickvals=[0,25,50,80,120,180],
                ticktext=["0","25 (WHO)","50","80 (NAAQS)","120","180"],
                tickfont=dict(size=10),
            ),
            showscale=True,
        ),
        text=annual["hover"],
        hovertemplate="%{text}<extra></extra>",
        hoverinfo="text",
    ))

    # ── TRACES 2–5: Seasonal layers (hidden by default) ─────────
    for s_name, smean in seasons.items():
        fig.add_trace(go.Scattermapbox(
            name=f"{s_name} mean",
            lat=smean["latitude"],
            lon=smean["longitude"],
            mode="markers",
            visible=False,
            marker=dict(
                size=6,
                color=smean["NO2_pred_ugm3"],
                colorscale=colorscale,
                cmin=0, cmax=vmax,
                opacity=0.85,
                showscale=False,
            ),
            text=smean["hover"],
            hovertemplate="%{text}<extra></extra>",
        ))

    # ── TRACE 6: NAAQS exceedance (hidden by default) ───────────
    exc_colorscale = [
        [0.0, "#ffffcc"],[0.25,"#fed976"],
        [0.5, "#fd8d3c"],[0.75,"#e31a1c"],[1.0,"#800026"]
    ]
    fig.add_trace(go.Scattermapbox(
        name="NAAQS exceedance %",
        lat=exceedance["latitude"],
        lon=exceedance["longitude"],
        mode="markers",
        visible=False,
        marker=dict(
            size=6,
            color=exceedance["pct_naaqs"],
            colorscale=exc_colorscale,
            cmin=0, cmax=100,
            opacity=0.85,
            showscale=False,
        ),
        text=exceedance["hover"],
        hovertemplate="%{text}<extra></extra>",
    ))

    # ── TRACE 7: WHO compliance zones ───────────────────────────
    compliance_colors = annual["status"].map({
        "Within WHO":    "#00aa00",
        "Exceeds WHO":   "#ff8800",
        "Exceeds NAAQS": "#cc0000",
    })
    fig.add_trace(go.Scattermapbox(
        name="WHO compliance",
        lat=annual["latitude"],
        lon=annual["longitude"],
        mode="markers",
        visible=False,
        marker=dict(
            size=6,
            color=compliance_colors,
            opacity=0.85,
            showscale=False,
        ),
        text=annual["hover"],
        hovertemplate="%{text}<extra></extra>",
    ))

    # ── TRACE 8: Ground stations ─────────────────────────────────
    st_lats = [v[0] for v in STATIONS.values()]
    st_lons = [v[1] for v in STATIONS.values()]
    st_text = [f"<b>📍 {k}</b><br>CPCB monitoring station"
               for k in STATIONS.keys()]

    fig.add_trace(go.Scattermapbox(
        name="Ground stations",
        lat=st_lats,
        lon=st_lons,
        mode="markers+text",
        visible=True,
        marker=dict(size=14, color="navy", symbol="circle"),
        text=[k for k in STATIONS.keys()],
        textposition="top right",
        textfont=dict(size=9, color="navy"),
        customdata=st_text,
        hovertemplate="%{customdata}<extra></extra>",
    ))

    # ── LAYOUT ──────────────────────────────────────────────────
    n_pixels = len(annual)
    mean_no2 = annual["NO2_pred_ugm3"].mean()
    pct_who  = (annual["NO2_pred_ugm3"] > WHO_LIMIT).mean()   * 100
    pct_naaqs= (annual["NO2_pred_ugm3"] > NAAQS_LIMIT).mean() * 100

    fig.update_layout(
        title=dict(
            text=(f"<b>Pune NO₂ Air Quality Map</b>  "
                  f"<span style='font-size:13px;color:#666'>"
                  f"Sentinel-5P Downscaling  |  {n_pixels:,} pixels  |  "
                  f"Mean: {mean_no2:.1f} µg/m³  |  "
                  f"Above WHO: {pct_who:.1f}%  |  "
                  f"Above NAAQS: {pct_naaqs:.1f}%</span>"),
            x=0.01,
            font=dict(size=16),
        ),
        mapbox=dict(
            style="open-street-map",   # free, no token needed
            center=dict(lat=18.52, lon=73.86),
            zoom=11,
        ),
        margin=dict(l=0, r=0, t=50, b=0),
        height=780,
        legend=dict(
            x=0.01, y=0.99,
            bgcolor="rgba(255,255,255,0.85)",
            bordercolor="#ccc",
            borderwidth=1,
            font=dict(size=12),
        ),
        # Buttons to switch map tile style
        updatemenus=[
            dict(
                type="buttons",
                direction="right",
                x=0.01, y=0.07,
                xanchor="left",
                bgcolor="rgba(255,255,255,0.85)",
                bordercolor="#ccc",
                font=dict(size=11),
                buttons=[
                    dict(label="Street",    method="relayout",
                         args=[{"mapbox.style":"open-street-map"}]),
                    dict(label="Satellite", method="relayout",
                         args=[{"mapbox.style":"satellite"}]),
                    dict(label="Terrain",   method="relayout",
                         args=[{"mapbox.style":"stamen-terrain"}]),
                    dict(label="Dark",      method="relayout",
                         args=[{"mapbox.style":"carto-darkmatter"}]),
                    dict(label="Light",     method="relayout",
                         args=[{"mapbox.style":"carto-positron"}]),
                ],
            ),
        ],
        # Annotations: WHO and NAAQS limit info
        annotations=[
            dict(
                x=0.01, y=0.01,
                xref="paper", yref="paper",
                text=(f"<span style='background:white;padding:4px'>"
                      f"🟠 WHO limit: {WHO_LIMIT} µg/m³  |  "
                      f"🔴 NAAQS limit: {NAAQS_LIMIT} µg/m³  |  "
                      f"Click pixels for details  |  "
                      f"Zoom/pan with mouse</span>"),
                showarrow=False,
                font=dict(size=11),
                bgcolor="rgba(255,255,255,0.8)",
                bordercolor="#ccc",
                borderwidth=1,
            )
        ],
    )

    return fig


# =============================================================
#  STEP 4 — SAVE HTML (memory-efficient write)
# =============================================================


def save_html(fig, path):
    print(f"\nWriting HTML → {path}")

    # Generate HTML string
    html_str = fig.to_html(
        include_plotlyjs="cdn",
        full_html=True,
        config={
            "scrollZoom": True,
            "displayModeBar": True,
            "modeBarButtonsToRemove": ["lasso2d","select2d"],
            "toImageButtonOptions": {
                "format": "png",
                "filename": "pune_no2_map",
                "height": 1200,
                "width": 1600,
                "scale": 2
            },
        },
    )

    # ✅ Force UTF-8 write (this fixes NO₂ issue)
    with open(path, "w", encoding="utf-8") as f:
        f.write(html_str)

    size_kb = os.path.getsize(path) / 1024
    print(f"  File size: {size_kb:.1f} KB  "
          f"({'small enough for email' if size_kb<5000 else 'large — share via Drive'})")
# =============================================================
#  BONUS: SEPARATE SEASON COMPARISON FIGURE
# =============================================================

def build_season_comparison(seasons):
    if not seasons:
        return None
    print("\nBuilding season comparison figure...")

    n = len(seasons)
    cols = 2
    rows = (n + 1) // 2

    colorscale = [
        [0.00,"#00E400"],[0.20,"#FFFF00"],
        [0.40,"#FF7E00"],[0.60,"#FF0000"],
        [0.80,"#8F3F97"],[1.00,"#7E0023"],
    ]

    vmax = max(
        float(smean["NO2_pred_ugm3"].quantile(0.97))
        for smean in seasons.values()
    )
    vmax = min(vmax, 200)

    fig = make_subplots(
        rows=rows, cols=cols,
        subplot_titles=list(seasons.keys()),
        specs=[[{"type":"mapbox"}]*cols]*rows,
        horizontal_spacing=0.02,
        vertical_spacing=0.05,
    )

    mapbox_configs = {}
    for i, (s_name, smean) in enumerate(seasons.items()):
        r = i // cols + 1
        c = i %  cols + 1
        mapbox_key = "mapbox" if (r==1 and c==1) else f"mapbox{i+1}"

        fig.add_trace(
            go.Scattermapbox(
                name=s_name,
                lat=smean["latitude"],
                lon=smean["longitude"],
                mode="markers",
                marker=dict(
                    size=5,
                    color=smean["NO2_pred_ugm3"],
                    colorscale=colorscale,
                    cmin=0, cmax=vmax,
                    opacity=0.85,
                    showscale=(i==0),
                    colorbar=dict(
                        title="NO₂ (µg/m³)",
                        thickness=12, len=0.4,
                        x=1.02,
                        tickvals=[0,25,80,vmax],
                        ticktext=["0","25 WHO","80 NAAQS",f"{vmax:.0f}"],
                        tickfont=dict(size=9),
                    ) if i==0 else {},
                ),
                text=smean["hover"],
                hovertemplate="%{text}<extra></extra>",
            ),
            row=r, col=c,
        )
        mapbox_configs[mapbox_key] = dict(
            style="open-street-map",
            center=dict(lat=18.52, lon=73.86),
            zoom=10,
        )

    fig.update_layout(
        title="<b>Pune NO₂ — Seasonal Comparison</b>",
        height=800 if rows==2 else 450,
        margin=dict(l=0,r=40,t=50,b=0),
        **{k: v for k,v in mapbox_configs.items()},
    )
    return fig


# =============================================================
#  MAIN
# =============================================================

if __name__ == "__main__":
    import time
    t0 = time.time()

    print("█"*60)
    print("  PUNE NO₂ — LIGHTWEIGHT INTERACTIVE MAP")
    print("  (Plotly WebGL — handles millions of points)")
    print("█"*60)

    # Load
    df = load_data()

    # Aggregate
    annual     = aggregate(df)
    seasons    = get_seasonal(df)
    exceedance = get_exceedance(df)

    # Main interactive map
    fig = build_figure(annual, seasons, exceedance, df)
    save_html(fig, OUTPUT_HTML)

    # Season comparison (separate file)




    fig_seasons = build_season_comparison(seasons)

    if fig_seasons:
        season_path = "map_outputs/pune_no2_seasons.html"

        html_str = fig_seasons.to_html(
            include_plotlyjs="cdn",
            full_html=True,
            config={"scrollZoom": True}
        )

        with open(season_path, "w", encoding="utf-8") as f:
            f.write(html_str)

        print(f"  Season comparison → {season_path}")

    elapsed = time.time() - t0
    print(f"\n  Total time: {elapsed:.1f}s")
    print("█"*60)
    print(f"  DONE — open in Chrome/Firefox:")
    print(f"    {OUTPUT_HTML}")
    print()
    print("  HOW TO USE:")
    print("  - Zoom/pan with mouse or touchpad")
    print("  - Click any pixel → NO₂ value + AQI status")
    print("  - Toggle layers in the legend (top-left)")
    print("  - Switch Street/Satellite/Terrain tiles (bottom-left buttons)")
    print("  - Camera icon (toolbar) → save as PNG")
    print("  - Hover over pixels for instant values")
    print("█"*60)

████████████████████████████████████████████████████████████
  PUNE NO₂ — LIGHTWEIGHT INTERACTIVE MAP
  (Plotly WebGL — handles millions of points)
████████████████████████████████████████████████████████████
Loading: data/output/all_predictions_500m.csv
  Rows: 2,536,270  Unique pixels: 44,491
Annual mean pixels: 44,491
NO2 range: 16.1 – 174.1 µg/m³

Building Plotly WebGL figure...

Writing HTML → map_outputs/pune_no2_map.html
  File size: 28889.9 KB  (large — share via Drive)

Building season comparison figure...
  Season comparison → map_outputs/pune_no2_seasons.html

  Total time: 104.5s
████████████████████████████████████████████████████████████
  DONE — open in Chrome/Firefox:
    map_outputs/pune_no2_map.html

  HOW TO USE:
  - Zoom/pan with mouse or touchpad
  - Click any pixel → NO₂ value + AQI status
  - Toggle layers in the legend (top-left)
  - Switch Street/Satellite/Terrain tiles (bottom-left buttons)
  - Camera icon (toolbar) → save as PNG
  - Hover over pixels for inst